# 🎙️ Pulsar Module 3: Daily Bangladeshi Hotwords & Zipformer ASR
### Contextual Hotword Biasing for 76 High-Frequency Smartphone Terms

**Problem Solved:**
Acoustic speech models trained on formal Bengali frequently confuse English loanwords with native phonetic roots (e.g. *ইমেইল* $\to$ *আবেল*, *কার্ড* $\to$ *কাঠ*).
By compiling an exact **SentencePiece BPE Hotwords Dictionary** and decoding via **Modified Beam Search with Hotword Biasing**, loanwords are accurately recognized at runtime with zero hallucinations.


In [ ]:
import os
import sherpa_onnx
import soundfile as sf
import sentencepiece as spm

MODEL_DIR = r"D:\Coding\Pulsar\prototype\python\kaggle_kernel\output\sherpa-onnx-streaming-zipformer-bn-vosk-2026-02-09"
BPE_MODEL = os.path.join(MODEL_DIR, "bpe.model")

sp = spm.SentencePieceProcessor()
sp.load(BPE_MODEL)
print(f"Loaded BPE Model with {sp.get_piece_size()} vocab pieces.")


In [ ]:
# Compile 76 High-Frequency Bangladeshi Mobile Hotwords
daily_words = [
    "ইমেইল", "ইমেইলটা", "মেইল", "মেইলটা", "বস", "বসের", "অফিস", "রিপোর্ট", "চেক",
    "হোয়াটসঅ্যাপ", "হোয়াটসঅ্যাপে", "মেসেজ", "মেসেজটা", "কল", "ফোন", "রিং",
    "পিডিএফ", "পিডিএফটা", "ইনভয়েস", "বিল", "প্রেসক্রিপশন", "এনআইডি", "টিকেট",
    "ছবি", "ছবিটা", "ফটো", "পিকচার", "ক্যামেরা", "গ্যালারি", "ডাউনলোড", "রেকর্ডিং",
    "ওপেন", "বাইর", "সার্চ", "গুগল", "ড্রাইভার", "ডাক্তার", "মিস্ত্রি", "আম্মু", "আব্বু"
]

hw_lines = []
for w in daily_words:
    pieces = sp.encode_as_pieces(w)
    hw_lines.append(f"{' '.join(pieces)} :2.5")

hw_path = r"D:\Coding\Pulsarangladesh_daily_hotwords.txt"
with open(hw_path, "w", encoding="utf-8") as f:
    f.write("\n".join(hw_lines))
print(f"Compiled {len(hw_lines)} hotwords to: {hw_path}")


In [ ]:
# Test ASR Recognition with Hotword Biasing
rec = sherpa_onnx.OnlineRecognizer.from_transducer(
    encoder=os.path.join(MODEL_DIR, "encoder.onnx"),
    decoder=os.path.join(MODEL_DIR, "decoder.onnx"),
    joiner=os.path.join(MODEL_DIR, "joiner.onnx"),
    tokens=os.path.join(MODEL_DIR, "tokens.txt"),
    num_threads=2,
    sample_rate=16000,
    feature_dim=80,
    decoding_method="modified_beam_search",
    max_active_paths=8,
    hotwords_file=hw_path,
    hotwords_score=2.5
)

audio_clip = r"D:\Coding\Pulsar\local\recordings\check boss's email_16k.wav"
data, sr = sf.read(audio_clip)

stream = rec.create_stream()
stream.accept_waveform(sr, data)
while rec.is_ready(stream):
    rec.decode_stream(stream)

print("Audio: check boss's email.aac")
print(f"Decoded with Hotwords Biasing: '{rec.get_result(stream)}'")
